# TMB association gene landscape: PQC vs. non-PQC

Volcano and ranked-beta plots using `tcga_all_genes_beta.tsv` and `pqc_annotation.tsv`.

**Color:** only PQC genes with FDR < 0.05 are highlighted in `#D55E00` (fig2c `pqc_axis_score`). All other genes are gray.

Summary counts are shown on the plot and saved to `gene_landscape_stats.txt`.

In [ ]:
import socket, os, sys

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.rcParams["pdf.fonttype"] = 42

hostname = socket.gethostname()
if "scg" in hostname:
    BASE_DIR = "/labs/mwinslow/Karen/protein-master/v2_pancan_batch_corrected"
else:
    BASE_DIR = os.path.expanduser("~/Documents/research/scg/protein-master/v2_pancan_batch_corrected")

FIG_DIR = f"{BASE_DIR}/local/figures"
BETA_PATH = f"{BASE_DIR}/analysis/00_reproduction/tilk_et_al_2023/gene_sets/bh/tcga_all_genes_beta.tsv"
PQC_PATH = f"{BASE_DIR}/data/corum/pqc_annotation.tsv"
OUT_DIR = f"{BASE_DIR}/results/figures/main/fig_tmb_gene_landscape"
os.makedirs(OUT_DIR, exist_ok=True)

sys.path.insert(0, FIG_DIR)
from palette import COLORS

PQC_SIG_COLOR = "#D55E00"      # fig2c pqc_axis_score; FDR < 0.05 PQC genes only
OTHER_COLOR = COLORS["ns"]
FDR_THRESH = 0.05

## Load and annotate genes

In [ ]:
beta = pd.read_csv(BETA_PATH, sep="\t").drop_duplicates(subset="gene", keep="first")

pqc_genes = pd.read_csv(PQC_PATH, sep="\t").drop_duplicates(subset="Hugo", keep="first")
pqc_genes.loc[pqc_genes["Hugo"] == "TOR4A", "Hugo"] = "C9orf167"
pqc_genes.loc[pqc_genes["Hugo"] == "FAXC", "Hugo"] = "C6orf168"
pqc_set = set(pqc_genes["Hugo"])

beta = beta.copy()
beta["is_pqc"] = beta["gene"].isin(pqc_set)
beta["neglog10_fdr"] = -np.log10(beta["fdr_bh"].clip(lower=1e-300))
beta["sig"] = beta["fdr_bh"] < FDR_THRESH
beta["sig_pos"] = beta["sig"] & (beta["estimate"] > 0)
beta["sig_neg"] = beta["sig"] & (beta["estimate"] < 0)
beta["plot_group"] = np.where(beta["is_pqc"] & beta["sig"], "pqc_sig", "other")
beta = beta.sort_values("estimate").reset_index(drop=True)
beta["rank"] = np.arange(1, len(beta) + 1)

pqc = beta[beta["is_pqc"]]
non_pqc = beta[~beta["is_pqc"]]

stats = {
    "n_total_genes": len(beta),
    "pqc_annotated": len(pqc_set),
    "pqc_with_beta": len(pqc),
    "pqc_beta_pos": int((pqc["estimate"] > 0).sum()),
    "pqc_beta_neg": int((pqc["estimate"] < 0).sum()),
    "pqc_fdr_sig": int(pqc["sig"].sum()),
    "pqc_fdr_sig_beta_pos": int(pqc["sig_pos"].sum()),
    "pqc_fdr_sig_beta_neg": int(pqc["sig_neg"].sum()),
    "non_pqc_with_beta": len(non_pqc),
    "non_pqc_beta_pos": int((non_pqc["estimate"] > 0).sum()),
    "non_pqc_beta_neg": int((non_pqc["estimate"] < 0).sum()),
    "non_pqc_fdr_sig": int(non_pqc["sig"].sum()),
    "non_pqc_fdr_sig_beta_pos": int(non_pqc["sig_pos"].sum()),
    "non_pqc_fdr_sig_beta_neg": int(non_pqc["sig_neg"].sum()),
}

stats_text = (
    f"PQC annotated: {stats['pqc_annotated']:,}\n"
    f"PQC with beta estimate: {stats['pqc_with_beta']:,}\n"
    f"PQC beta > 0: {stats['pqc_beta_pos']:,}; beta < 0: {stats['pqc_beta_neg']:,}\n"
    f"PQC FDR < {FDR_THRESH}: {stats['pqc_fdr_sig']:,}\n"
    f"PQC FDR < {FDR_THRESH}, beta > 0: {stats['pqc_fdr_sig_beta_pos']:,}\n"
    f"PQC FDR < {FDR_THRESH}, beta < 0: {stats['pqc_fdr_sig_beta_neg']:,}\n"
    f"\n"
    f"Non-PQC with beta estimate: {stats['non_pqc_with_beta']:,}\n"
    f"Non-PQC beta > 0: {stats['non_pqc_beta_pos']:,}; beta < 0: {stats['non_pqc_beta_neg']:,}\n"
    f"Non-PQC FDR < {FDR_THRESH}: {stats['non_pqc_fdr_sig']:,}\n"
    f"Non-PQC FDR < {FDR_THRESH}, beta > 0: {stats['non_pqc_fdr_sig_beta_pos']:,}\n"
    f"Non-PQC FDR < {FDR_THRESH}, beta < 0: {stats['non_pqc_fdr_sig_beta_neg']:,}"
)

with open(f"{OUT_DIR}/gene_landscape_stats.txt", "w") as f:
    f.write("TMB gene landscape summary\n")
    f.write("=" * 40 + "\n")
    for key, val in stats.items():
        f.write(f"{key}: {val:,}\n")

print(stats_text)
print(f"\nSaved: {OUT_DIR}/gene_landscape_stats.txt")

## Volcano plot

In [ ]:
GROUP_STYLE = {
    "other": {"color": OTHER_COLOR, "size": 6, "alpha": 0.2, "edgecolors": "none", "linewidths": 0, "zorder": 1},
    "pqc_sig": {"color": PQC_SIG_COLOR, "size": 18, "alpha": 0.9, "edgecolors": "white", "linewidths": 0.4, "zorder": 3},
}


def plot_beta_landscape(ax, x_col, y_col):
    for group in ["other", "pqc_sig"]:
        sub = beta[beta["plot_group"] == group]
        style = GROUP_STYLE[group]
        ax.scatter(
            sub[x_col], sub[y_col],
            s=style["size"], alpha=style["alpha"], color=style["color"],
            edgecolors=style["edgecolors"], linewidths=style["linewidths"],
            rasterized=True, zorder=style["zorder"],
        )


fig, ax = plt.subplots(figsize=(8, 7))
plot_beta_landscape(ax, "estimate", "neglog10_fdr")

ax.axvline(0, color=COLORS["dark_gray"], lw=0.8, ls="--")
ax.axhline(-np.log10(FDR_THRESH), color=COLORS["dark_gray"], lw=0.8, ls="--")
ax.set_xlabel("TMB association (beta estimate)")
ax.set_ylabel(r"$-\log_{10}$(FDR)")
ax.set_title("Volcano plot: all genes", fontsize=12)
ax.text(
    0.02, 0.98, stats_text,
    transform=ax.transAxes, va="top", ha="left", fontsize=8,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=COLORS["light_gray"], alpha=0.95),
)

legend_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=OTHER_COLOR, markersize=5, label="All other genes"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=PQC_SIG_COLOR, markersize=7, label=f"PQC, FDR<{FDR_THRESH} (n={stats['pqc_fdr_sig']:,})"),
]
ax.legend(handles=legend_handles, frameon=False, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/volcano_pqc.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Ranked-beta plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
plot_beta_landscape(ax, "rank", "estimate")

ax.axhline(0, color=COLORS["dark_gray"], lw=0.8, ls="--")
ax.set_xlabel("Rank (low to high beta)")
ax.set_ylabel("TMB association (beta estimate)")
ax.set_title("Ranked-beta plot: all genes", fontsize=12)
ax.text(
    0.02, 0.98, stats_text,
    transform=ax.transAxes, va="top", ha="left", fontsize=8.5,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=COLORS["light_gray"], alpha=0.95),
)
ax.legend(handles=legend_handles, frameon=False, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ranked_beta_pqc.pdf", dpi=300, bbox_inches="tight")
plt.show()